In [ ]:
import os
import json
import base64
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

: 

In [ ]:
load_dotenv()

In [ ]:
# Path.cwd() gets the current working directory (the demo/ folder when run from there).
# .parent moves one level up to the repository root.
PROJECT_ROOT = Path.cwd().parent

# Use the / operator to join paths reliably across operating systems.
DATASET_PATH = PROJECT_ROOT / "examples" / "HarmfulMemes-tiny"
JSON_FILE    = "train-tiny.jsonl"

In [ ]:
def load_top_n_samples(file_path: Path, n: int = 8) -> list:
    """Read the first n JSONL records from file_path."""
    samples = []
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            samples.append(json.loads(line))
    return samples

In [ ]:
def get_multimodal_data(sample: dict, root_path: Path) -> tuple:
    """Return (text, image_path) for a single dataset record."""
    text     = sample["text"]
    img_path = root_path / sample["img"]
    return text, img_path

In [ ]:
def encode_image(path: Path) -> str:
    """Base64-encode an image file for inline API submission."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [ ]:
def build_multimodal_message(text: str, img_base64: str) -> HumanMessage:
    """Construct a multimodal HumanMessage for LangChain."""
    return HumanMessage(content=[
        {
            "type": "text",
            "text": (
                "Task: Analyze if the following social media post contains hateful speech.\n"
                f"Text content: '{text}'\n"
                "Requirement: Consider both text and image context. "
                "Output <hateful> or <not hateful> with a brief explanation."
            ),
        },
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img_base64}"},
        },
    ])

In [ ]:
chat = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL") or "https://api.openai.com/v1",
)

json_path = DATASET_PATH / JSON_FILE
samples   = load_top_n_samples(json_path, n=8)

for sample in samples:
    text, img_path = get_multimodal_data(sample, DATASET_PATH)
    b64 = encode_image(img_path)
    msg = build_multimodal_message(text, b64)
    res = chat.invoke([msg])
    print(res.content)